
# Del análisis bivariante al análisis multivariante

| Elemento | Detalle |
|---|---|
| Modalidad | Clase teórico-práctica |
| Dataset | Sleep Health and Lifestyle (salud y hábitos de sueño) |


## Propósito

En la sesión anterior se estudiaron las variables de forma individual (análisis univariante) y las relaciones entre pares de variables (análisis bivariante). En esta sesión el análisis se amplía al estudio de **varias variables de forma simultánea**.

El propósito no es generar una mayor cantidad de gráficos, sino responder preguntas analíticas más completas, justificando qué técnica es adecuada para cada tipo de variable y qué conclusiones permite sostener la evidencia disponible.

**Ruta de trabajo:** pregunta, variables, técnica, evidencia, interpretación, limitaciones y conclusión.

# 2. Marco conceptual: niveles del análisis exploratorio

El análisis exploratorio de datos, formalizado por John Tukey (1977), busca comprender la estructura de los datos antes de formular modelos o realizar inferencias formales. Según el número de variables involucradas se distinguen tres niveles.

| Nivel | Número de variables | Pregunta ilustrativa | Objetivo principal |
|---|---:|---|---|
| Univariante | 1 | ¿Cómo se distribuye el nivel de estrés? | Describir |
| Bivariante | 2 | ¿Qué asociación existe entre duración y calidad del sueño? | Relacionar |
| Multivariante | 3 o más | ¿Cómo se relacionan duración, calidad y estrés de forma conjunta? | Contextualizar y sintetizar |

Una tercera variable puede cumplir distintas funciones:

- **Contexto:** permite observar si una relación se mantiene en distintos subgrupos.
- **Confusión:** puede generar o ocultar una asociación aparente entre otras dos variables.
- **Resumen:** varias variables pueden condensarse en pocas dimensiones (por ejemplo, mediante PCA).

> Incluir más variables no equivale a realizar un mejor análisis. Las variables deben seleccionarse porque contribuyen a responder una pregunta concreta, y cada variable adicional incrementa la complejidad de la interpretación.

# 3. Selección de la técnica según el tipo de variable

La técnica apropiada depende de la escala de medición de las variables. Aplicar una técnica inadecuada produce resultados numéricamente válidos pero conceptualmente erróneos.

| Variables involucradas | Pregunta típica | Medidas y pruebas | Visualización recomendada |
|---|---|---|---|
| Numérica y numérica | ¿Varían conjuntamente? | Covarianza, correlación de Pearson, correlación de Spearman | Diagrama de dispersión, mapa de calor, pairplot |
| Numérica y categórica | ¿El nivel de la variable difiere entre grupos? | Media, mediana y desviación estándar por grupo | Diagrama de cajas, violín |
| Categórica y categórica | ¿Las categorías se asocian entre sí? | Tabla de contingencia, chi-cuadrado, V de Cramér | Barras apiladas, mapa de calor de proporciones |
| Tres o más variables | ¿El patrón se mantiene al considerar otras variables? | Correlación parcial, análisis por subgrupos, PCA | Color, tamaño, forma, facetas |

### Escalas de medición

| Escala | Características | Ejemplo en el dataset |
|---|---|---|
| Nominal | Categorías sin orden | `Gender`, `Sleep Disorder` |
| Ordinal | Categorías con orden, distancias no necesariamente iguales | `BMI Category`, `Quality of Sleep` (escala 1 a 10) |
| Intervalo / razón | Valores numéricos con distancias interpretables | `Age`, `Sleep Duration`, `Daily Steps` |

La escala determina qué operaciones tienen sentido. Por ejemplo, el coeficiente de Pearson supone distancias interpretables entre valores, supuesto discutible en escalas ordinales de pocos niveles. Esta cuestión se retoma en la sección 11.

# 4. Preparación del entorno y carga de datos

Se importan las bibliotecas necesarias y se establece un estilo gráfico uniforme. La carga del archivo utiliza `keep_default_na=False` por una razón técnica: en este dataset, la categoría `None` de la variable `Sleep Disorder` significa "sin trastorno del sueño", pero `pandas` la interpreta por defecto como valor faltante. Si no se controla este comportamiento, una categoría legítima se perdería del análisis.

In [ ]:
# Instalación (solo si hace falta):
# %pip install pandas numpy matplotlib seaborn scipy scikit-learn

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

In [ ]:
# Rutas candidatas del archivo de datos.
rutas_posibles = [
    Path("../datasets/sleep_health_lifestyle.csv"),
    Path("datasets/sleep_health_lifestyle.csv"),
    Path("sleep_health_lifestyle.csv"),
]
ruta = next((r for r in rutas_posibles if r.exists()), None)
if ruta is None:
    raise FileNotFoundError(
        "No se encontró sleep_health_lifestyle.csv. Verificar la ruta en 'rutas_posibles'."
    )

# keep_default_na=False evita que la categoría 'None' se convierta en NaN.
# na_values=[""] conserva como faltantes únicamente las celdas vacías.
df = pd.read_csv(ruta, keep_default_na=False, na_values=[""])
print(f"Archivo cargado: {ruta}")
print(f"Dimensiones: {df.shape[0]} registros y {df.shape[1]} columnas")
df.head()

# 5. Ficha del dataset y diccionario de variables

Todo análisis debe iniciar documentando el origen, la unidad de observación y el significado de cada variable. Sin esta información, la interpretación de resultados carece de fundamento.

| Aspecto | Descripción |
|---|---|
| Unidad de observación | Una persona (un registro por individuo) |
| Origen | Dataset público de salud y hábitos de sueño, de uso didáctico |
| Naturaleza de los datos | Con alta probabilidad sintéticos o simulados; no representan a una población real |
| Implicación | Los hallazgos sirven para practicar la metodología, no para sustentar afirmaciones sobre la salud de personas reales |

| Variable | Tipo | Descripción |
|---|---|---|
| `Gender` | Nominal | Género declarado |
| `Age` | Numérica | Edad en años |
| `Occupation` | Nominal | Ocupación |
| `Sleep Duration` | Numérica | Horas de sueño por día |
| `Quality of Sleep` | Ordinal (1 a 10) | Calidad del sueño autoevaluada |
| `Physical Activity Level` | Numérica | Minutos de actividad física por día |
| `Stress Level` | Ordinal (1 a 10) | Nivel de estrés autoevaluado |
| `BMI Category` | Ordinal | Categoría de índice de masa corporal |
| `Blood Pressure` | Texto | Presión arterial en formato sistólica/diastólica |
| `Heart Rate` | Numérica | Frecuencia cardíaca en reposo (latidos por minuto) |
| `Daily Steps` | Numérica | Pasos diarios |
| `Sleep Disorder` | Nominal | Trastorno del sueño: `None`, `Insomnia` o `Sleep Apnea` |

**Pregunta de reflexión.** Si los datos son en buena medida simulados, ¿qué tipo de conclusiones podrían sostenerse y cuáles no? La respuesta debe quedar registrada como parte de las limitaciones del análisis.

# 6. Auditoría de calidad de los datos

Antes de calcular asociaciones se verifica la calidad de los datos. Las asociaciones calculadas sobre datos mal tipificados, con faltantes no tratados o con categorías inconsistentes pueden conducir a conclusiones erróneas.

Los controles mínimos son:

1. **Estructura y tipos de datos:** verificar que cada columna tenga el tipo esperado.
2. **Valores faltantes:** cuantificarlos y decidir un tratamiento justificado.
3. **Duplicados:** identificarlos y evaluar si son errores o coincidencias legítimas.
4. **Rangos y categorías:** detectar valores imposibles y etiquetas inconsistentes.

In [ ]:
df.info()

In [ ]:
# Valores faltantes por columna y registros duplicados.
faltantes = df.isna().sum().rename("Faltantes").to_frame()
faltantes["Porcentaje"] = (faltantes["Faltantes"] / len(df) * 100).round(2)
display(faltantes)

print(f"Registros duplicados exactos: {df.duplicated().sum()}")

### Criterio sobre los duplicados

Un duplicado exacto en datos de encuesta no es necesariamente un error: dos personas distintas pueden coincidir en todas sus respuestas, sobre todo cuando varias variables tienen pocos valores posibles. Eliminar duplicados sin criterio puede distorsionar la distribución real. La decisión debe documentarse y, cuando no existe un identificador único que permita verificar el error, lo prudente es conservarlos y declarar la limitación.

In [ ]:
# Resumen estadístico de las variables numéricas.
df.describe().T.round(2)

In [ ]:
# Revisión de categorías en variables cualitativas.
for columna in ["Gender", "BMI Category", "Sleep Disorder"]:
    print(f"--- {columna} ---")
    print(df[columna].value_counts(dropna=False), end="\n\n")

### Armonización de categorías

Si en `BMI Category` coexisten las etiquetas `Normal` y `Normal Weight`, ambas representan la misma categoría clínica. Mantenerlas separadas fragmentaría el grupo y debilitaría cualquier comparación. La armonización de etiquetas es una decisión de limpieza que debe quedar explícita en el análisis.

In [ ]:
# Armonización de etiquetas equivalentes (no tiene efecto si solo existe una de ellas).
df["BMI Category"] = df["BMI Category"].replace({"Normal Weight": "Normal"})
df["BMI Category"].value_counts()

# 7. Naturaleza de las variables ordinales

`Quality of Sleep` y `Stress Level` son escalas de autoevaluación con pocos valores enteros posibles. Estrictamente son variables **ordinales**: se sabe que 8 es mayor que 7, pero no está garantizado que la diferencia entre 7 y 8 equivalga a la diferencia entre 3 y 4.

Esta característica tiene dos consecuencias para el análisis:

- Los diagramas de dispersión presentan **superposición de puntos** (muchos registros comparten las mismas coordenadas), lo que oculta la densidad real.
- El coeficiente de Pearson, que supone relaciones lineales entre variables con distancias interpretables, puede no ser la medida más adecuada. El coeficiente de Spearman, basado en rangos, es una alternativa pertinente.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

for eje, variable in zip(ejes, ["Quality of Sleep", "Stress Level"]):
    conteo = df[variable].value_counts().sort_index()
    eje.bar(conteo.index.astype(str), conteo.values, color="#4C78A8")
    eje.set_title(f"Distribución de {variable}")
    eje.set_xlabel("Valor de la escala")
    eje.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

# 8. Primera exploración multivariante

Pregunta exploratoria:

> **¿La relación entre duración y calidad del sueño parece diferente cuando se considera el nivel de estrés?**

| Rol | Variable |
|---|---|
| Eje X | `Sleep Duration` |
| Eje Y | `Quality of Sleep` |
| Variable de contexto (color) | `Stress Level` |

Dado que las escalas son discretas, se agrega una pequeña dispersión aleatoria (`jitter`) con fines exclusivamente visuales, para que los puntos superpuestos puedan distinguirse. Los cálculos posteriores utilizan siempre los valores originales.

In [ ]:
rng = np.random.default_rng(42)
jitter_y = rng.uniform(-0.15, 0.15, size=len(df))

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=df["Sleep Duration"],
    y=df["Quality of Sleep"] + jitter_y,
    hue=df["Stress Level"],
    palette="viridis",
    alpha=0.75,
)
plt.title("Duración, calidad del sueño y nivel de estrés")
plt.xlabel("Duración del sueño (horas)")
plt.ylabel("Calidad del sueño (con dispersión visual)")
plt.legend(title="Nivel de estrés", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Interpretación guiada

La lectura de un diagrama multivariante debe realizarse de forma sistemática:

1. **Tendencia general:** ¿se observa una dirección predominante entre duración y calidad?
2. **Forma de la relación:** ¿es aproximadamente lineal, curva o presenta saltos?
3. **Dispersión:** ¿los puntos se concentran cerca de la tendencia o se dispersan ampliamente?
4. **Agrupamientos:** ¿existen conglomerados de observaciones con características similares?
5. **Valores atípicos:** ¿hay observaciones alejadas del patrón general?
6. **Aporte de la tercera variable:** ¿los colores se organizan de forma ordenada (por ejemplo, de un extremo a otro del gráfico) o se mezclan sin patrón?

> **Precaución.** Que tres variables aparezcan visualmente relacionadas no demuestra causalidad. El análisis exploratorio permite describir patrones y formular hipótesis, no confirmarlas.

# 9. Covarianza y coeficiente de correlación

### Covarianza

La covarianza muestral mide si dos variables tienden a variar en el mismo sentido:

$$ \operatorname{cov}(X, Y) = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y}) $$

- Si los valores altos de $X$ coinciden con valores altos de $Y$, los productos son mayoritariamente positivos y la covarianza es positiva.
- Si los valores altos de $X$ coinciden con valores bajos de $Y$, la covarianza es negativa.

**Limitación:** su magnitud depende de las unidades de medida. Una covarianza de 150 entre pasos diarios y minutos de actividad no es comparable con una de 0.9 entre horas de sueño y calidad del sueño.

### Coeficiente de correlación de Pearson

El coeficiente de Pearson corrige esa limitación dividiendo la covarianza entre el producto de las desviaciones estándar:

$$ r_{xy} = \frac{\operatorname{cov}(X, Y)}{s_X \, s_Y} $$

El resultado es adimensional y se encuentra siempre entre $-1$ y $+1$. Equivale a la covarianza de las variables estandarizadas (puntuaciones z).

### Coeficiente de determinación

El cuadrado del coeficiente, $r^2$, indica la proporción de la variabilidad de una variable que es compartida linealmente con la otra. Una correlación de $r = 0.50$ implica $r^2 = 0.25$: solo el 25 % de la variabilidad es compartida, aun cuando la correlación se describa como moderada.

In [ ]:
# Verificación numérica de la relación entre covarianza y correlación.
x = df["Sleep Duration"]
y = df["Quality of Sleep"]

covarianza = x.cov(y)
r_manual = covarianza / (x.std() * y.std())
r_pandas = x.corr(y)

print(f"Covarianza (Sleep Duration, Quality of Sleep): {covarianza:.4f}")
print(f"r calculado manualmente: {r_manual:.4f}")
print(f"r calculado con pandas:  {r_pandas:.4f}")
print(f"r al cuadrado:           {r_pandas ** 2:.4f}")

## 9.1 Dirección e intensidad

- Si $r > 0$: asociación lineal **positiva** (ambas variables tienden a aumentar juntas).
- Si $r < 0$: asociación lineal **negativa** (cuando una aumenta, la otra tiende a disminuir).
- Si $r \approx 0$: asociación lineal **muy débil o inexistente**. No implica ausencia de relación: puede existir una relación no lineal.

La **dirección** se determina por el signo, y la **intensidad** por el valor absoluto $|r|$. Por ejemplo, $+0.80$ y $-0.80$ tienen la misma intensidad con direcciones opuestas.

## 9.2 Escala orientativa de intensidad

No existe una escala universal. Cohen (1988) propuso referencias para ciencias del comportamiento, y distintas disciplinas adoptan umbrales propios. Para esta asignatura se utiliza la siguiente **guía didáctica**:

| Valor absoluto de r | Interpretación orientativa |
|---:|---|
| 0.00 a 0.19 | Muy débil |
| 0.20 a 0.39 | Débil |
| 0.40 a 0.59 | Moderada |
| 0.60 a 0.79 | Fuerte |
| 0.80 a 1.00 | Muy fuerte |

| r | Dirección | Intensidad |
|---:|---|---|
| +0.91 | Positiva | Muy fuerte |
| +0.67 | Positiva | Fuerte |
| +0.43 | Positiva | Moderada |
| +0.24 | Positiva | Débil |
| +0.05 | Positiva | Muy débil |
| -0.36 | Negativa | Débil |
| -0.72 | Negativa | Fuerte |
| -0.88 | Negativa | Muy fuerte |

Estos puntos de corte son **convenciones orientativas**, no leyes estadísticas. Su relevancia depende del campo de estudio, de la calidad de los datos y del objetivo del análisis. Una correlación de 0.30 puede ser trivial en ingeniería y muy relevante en epidemiología.

In [ ]:
def clasificar_intensidad(r):
    """Clasifica |r| según la escala orientativa utilizada en la asignatura."""
    a = abs(r)
    if a < 0.20:
        return "Muy débil"
    if a < 0.40:
        return "Débil"
    if a < 0.60:
        return "Moderada"
    if a < 0.80:
        return "Fuerte"
    return "Muy fuerte"


def clasificar_direccion(r):
    if abs(r) < 0.05:
        return "Prácticamente nula"
    return "Positiva" if r > 0 else "Negativa"


for valor in [0.78, -0.83, 0.31, -0.08, 0.52]:
    print(f"r = {valor:+.2f}  ->  {clasificar_direccion(valor)}, {clasificar_intensidad(valor)}")

### Ejercicio de interpretación

Antes de ejecutar la celda anterior, se solicita clasificar manualmente las siguientes asociaciones (dirección e intensidad orientativa) y registrar la respuesta:

1. $r = 0.78$
2. $r = -0.83$
3. $r = 0.31$
4. $r = -0.08$
5. $r = 0.52$

Posteriormente, explicar por qué $+0.78$ y $-0.78$ tienen la misma intensidad con direcciones opuestas, y por qué un coeficiente negativo no implica que la relación sea "mala".

# 10. Matriz de correlación

Cuando se dispone de varias variables numéricas, las asociaciones lineales de todos los pares se resumen en una **matriz de correlación**. Esta matriz presenta las siguientes propiedades:

- Es **cuadrada** y **simétrica**: la correlación de X con Y es igual a la de Y con X.
- La **diagonal principal** vale 1, porque cada variable se correlaciona perfectamente consigo misma.
- Para $p$ variables contiene $p(p-1)/2$ pares distintos. Con 7 variables son 21 pares, lo que anticipa el problema de las comparaciones múltiples (sección 13).

In [ ]:
variables_numericas = [
    "Age",
    "Sleep Duration",
    "Quality of Sleep",
    "Physical Activity Level",
    "Stress Level",
    "Heart Rate",
    "Daily Steps",
]

correlaciones = df[variables_numericas].corr(method="pearson")
correlaciones.round(2)

## 10.1 Mapa de calor

La tabla resulta difícil de inspeccionar cuando aumenta el número de variables. El mapa de calor codifica la intensidad con color. Dado que la matriz es simétrica, se muestra únicamente el triángulo inferior para evitar redundancia visual.

In [ ]:
mascara = np.triu(np.ones_like(correlaciones, dtype=bool))

plt.figure(figsize=(10, 7.5))
sns.heatmap(
    correlaciones,
    mask=mascara,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Coeficiente de Pearson (r)"},
)
plt.title("Matriz de correlación de Pearson (triángulo inferior)")
plt.tight_layout()
plt.show()

### Lectura sistemática de la matriz

1. Examinar el **signo** (dirección).
2. Evaluar la **magnitud** (intensidad).
3. Descartar la diagonal.
4. Seleccionar los pares relevantes para la pregunta de investigación.
5. **Contrastar siempre la cifra con un gráfico**, porque un mismo valor de $r$ puede corresponder a patrones muy distintos (sección 12).

Un buen análisis no describe todas las celdas: selecciona los resultados pertinentes y los interpreta.

In [ ]:
def pares_unicos(matriz, nombre="valor"):
    """Devuelve los pares distintos de una matriz simétrica (triángulo inferior)."""
    mascara_inferior = np.tril(np.ones(matriz.shape, dtype=bool), k=-1)
    serie = matriz.where(mascara_inferior).stack()
    serie.name = nombre
    return serie


ranking = pares_unicos(correlaciones, "r").reset_index()
ranking.columns = ["Variable 1", "Variable 2", "r"]
ranking["Dirección"] = ranking["r"].apply(clasificar_direccion)
ranking["Intensidad"] = ranking["r"].apply(clasificar_intensidad)
ranking["r al cuadrado"] = ranking["r"] ** 2
ranking = ranking.reindex(ranking["r"].abs().sort_values(ascending=False).index)

ranking.head(10).round(3).reset_index(drop=True)

**Actividad.** A partir del ranking anterior, seleccionar los tres pares más relevantes para una pregunta sobre bienestar y sueño. Para cada uno, redactar una oración que indique dirección, intensidad y proporción de variabilidad compartida ($r^2$), y señalar qué gráfico sería necesario para verificar el resultado.

# 11. Pearson frente a Spearman

| Aspecto | Pearson | Spearman |
|---|---|---|
| Mide | Asociación **lineal** | Asociación **monótona** (la variable aumenta o disminuye de forma consistente, aunque no sea en línea recta) |
| Base de cálculo | Valores originales | Rangos (posiciones ordenadas) |
| Escala requerida | Intervalo o razón | Ordinal, intervalo o razón |
| Sensibilidad a atípicos | Alta | Menor |

El coeficiente de Spearman equivale al de Pearson aplicado a los rangos de las observaciones. Cuando ambos coeficientes difieren de manera apreciable, es un indicio de relaciones no lineales, de atípicos influyentes o de efectos de escalas discretas, y justifica una inspección gráfica.

In [ ]:
pearson = df[variables_numericas].corr(method="pearson")
spearman = df[variables_numericas].corr(method="spearman")

comparacion = pd.concat(
    [pares_unicos(pearson, "Pearson"), pares_unicos(spearman, "Spearman")],
    axis=1,
)
comparacion["Diferencia absoluta"] = (comparacion["Pearson"] - comparacion["Spearman"]).abs()

comparacion.sort_values("Diferencia absoluta", ascending=False).round(3).head(8)

### Interpretación

- Cuando las dos medidas son similares, la asociación es consistente con una relación aproximadamente lineal y monótona.
- Cuando difieren, se debe examinar el diagrama de dispersión correspondiente para determinar si la causa es una relación curva, la presencia de atípicos o la discretización de la escala.
- Para `Quality of Sleep` y `Stress Level`, que son ordinales, Spearman es conceptualmente más defendible, aunque en la práctica ambos coeficientes suelen coincidir en conclusiones generales.

# 12. El cuarteto de Anscombe: por qué siempre se debe graficar

Francis Anscombe (1973) construyó cuatro conjuntos de datos con **estadísticos prácticamente idénticos** (medias, varianzas, correlación y recta de regresión) pero con estructuras completamente distintas. Es el argumento clásico a favor de la visualización como parte obligatoria del análisis.

In [ ]:
x_comun = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = pd.concat(
    [
        pd.DataFrame({"conjunto": "I",   "x": x_comun, "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]}),
        pd.DataFrame({"conjunto": "II",  "x": x_comun, "y": [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]}),
        pd.DataFrame({"conjunto": "III", "x": x_comun, "y": [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]}),
        pd.DataFrame({"conjunto": "IV",  "x": [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], "y": [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]}),
    ],
    ignore_index=True,
)

resumen_anscombe = pd.DataFrame(
    {
        nombre: {
            "media x": g["x"].mean(),
            "media y": g["y"].mean(),
            "varianza x": g["x"].var(),
            "varianza y": g["y"].var(),
            "r": g["x"].corr(g["y"]),
        }
        for nombre, g in anscombe.groupby("conjunto")
    }
).T
resumen_anscombe.round(2)

In [ ]:
g = sns.lmplot(
    data=anscombe,
    x="x",
    y="y",
    col="conjunto",
    col_wrap=2,
    height=3.4,
    ci=None,
    scatter_kws={"s": 45, "color": "#4C78A8"},
    line_kws={"color": "#E45756"},
)
g.figure.suptitle("Cuarteto de Anscombe: mismos estadísticos, estructuras distintas", y=1.02)
plt.show()

### Análisis

La tabla muestra estadísticos casi indistinguibles; los gráficos revelan cuatro situaciones diferentes:

| Conjunto | Estructura | Consecuencia para el análisis |
|---|---|---|
| I | Relación lineal con dispersión moderada | El coeficiente de Pearson es apropiado |
| II | Relación curva | Pearson subestima una relación que existe pero no es lineal |
| III | Relación lineal con un atípico | Un único punto distorsiona la pendiente y el coeficiente |
| IV | Un punto con alta influencia | Toda la correlación depende de una sola observación |

**Conclusión metodológica.** Un coeficiente de correlación resume una relación en un solo número y, por lo tanto, pierde información. Debe acompañarse siempre de la inspección gráfica.

# 13. Inferencia sobre la correlación

Un coeficiente calculado en una muestra es una **estimación** del valor poblacional. Dos herramientas permiten expresar su incertidumbre.

### Valor p

La prueba de hipótesis para la correlación plantea:

- $H_0$: la correlación poblacional es cero ($\rho = 0$).
- $H_1$: la correlación poblacional es distinta de cero.

Un valor p pequeño indica que los datos serían poco probables si no existiera correlación lineal en la población. **No indica que la relación sea fuerte ni importante.**

### Intervalo de confianza

Se calcula mediante la transformación de Fisher ($z = \operatorname{arctanh}(r)$, con error estándar $1/\sqrt{n-3}$) y se retransforma a la escala original. Un intervalo estrecho indica una estimación precisa; un intervalo que incluye valores de signos opuestos indica que la dirección misma es incierta.

In [ ]:
def resumen_correlacion(x, y, alfa=0.05):
    """Devuelve r, r^2, valor p e intervalo de confianza (transformación de Fisher)."""
    datos = pd.concat([x, y], axis=1).dropna()
    n = len(datos)
    r, p = stats.pearsonr(datos.iloc[:, 0], datos.iloc[:, 1])
    z = np.arctanh(r)
    error = 1 / np.sqrt(n - 3)
    z_critico = stats.norm.ppf(1 - alfa / 2)
    return {
        "n": n,
        "r": r,
        "r al cuadrado": r ** 2,
        "valor p": p,
        "IC 95% inferior": np.tanh(z - z_critico * error),
        "IC 95% superior": np.tanh(z + z_critico * error),
    }


pares_a_evaluar = [
    ("Sleep Duration", "Quality of Sleep"),
    ("Stress Level", "Quality of Sleep"),
    ("Physical Activity Level", "Quality of Sleep"),
    ("Age", "Heart Rate"),
    ("Daily Steps", "Physical Activity Level"),
]

filas = []
for a, b in pares_a_evaluar:
    fila = {"Variable 1": a, "Variable 2": b}
    fila.update(resumen_correlacion(df[a], df[b]))
    filas.append(fila)

tabla_inferencia = pd.DataFrame(filas)
tabla_inferencia["valor p"] = tabla_inferencia["valor p"].map(lambda v: f"{v:.2e}")
tabla_inferencia.round(3)

## 13.1 Significancia estadística frente a relevancia práctica

Con muestras grandes, correlaciones muy pequeñas resultan estadísticamente significativas. La siguiente simulación genera datos con una correlación poblacional **fija y débil** ($\rho = 0.10$) y muestra cómo el valor p disminuye únicamente por aumentar el tamaño de muestra, sin que la relación se vuelva más importante.

In [ ]:
rng = np.random.default_rng(2026)
rho = 0.10
covarianza_teorica = [[1, rho], [rho, 1]]

filas = []
for n in [30, 100, 374, 2000, 10000]:
    muestra = rng.multivariate_normal([0, 0], covarianza_teorica, size=n)
    r, p = stats.pearsonr(muestra[:, 0], muestra[:, 1])
    filas.append({"Tamaño de muestra": n, "r observado": r, "r al cuadrado": r ** 2, "valor p": p})

pd.DataFrame(filas).round(4)

### Reglas de interpretación

1. **Significancia estadística no es relevancia práctica.** La magnitud del efecto ($r$, $r^2$) y el intervalo de confianza informan sobre la importancia; el valor p solo informa sobre la compatibilidad con $H_0$.
2. **Comparaciones múltiples.** Si se examinan 21 pares al nivel de 5 %, se espera que aproximadamente uno resulte "significativo" por azar. Un ajuste conservador es el de Bonferroni: $\alpha_{ajustado} = \alpha / m$, donde $m$ es el número de pruebas.
3. **Muestras no aleatorias.** Los valores p suponen una muestra aleatoria de la población de interés. En un dataset de uso didáctico esa condición no se cumple, por lo que la inferencia debe leerse como ilustrativa.

# 14. Correlación, causalidad y variables de confusión

Una correlación fuerte entre `Sleep Duration` y `Quality of Sleep` permite afirmar:

> Existe una asociación lineal positiva en los registros analizados.

No permite afirmar automáticamente:

> Dormir más provoca una mejor calidad de sueño.

La correlación por sí sola no establece causalidad, dirección causal, ausencia de variables de confusión, mecanismo ni representatividad de la muestra.

### Explicaciones alternativas para una asociación entre X e Y

| Explicación | Descripción | Ejemplo ilustrativo |
|---|---|---|
| X causa Y | Relación causal directa | Mayor exposición a un factor genera un efecto |
| Y causa X | Causalidad inversa | El estrés elevado podría reducir las horas de sueño, y no solo al contrario |
| Z causa X e Y | **Variable de confusión** | La edad influye tanto en la actividad física como en la frecuencia cardíaca |
| Azar | Coincidencia en una muestra | Correlaciones espurias con pocos datos o muchas comparaciones |
| Selección | La muestra no es representativa | Solo se encuestó a un grupo particular |

```text
             Variable Z (confusión)
              /                \
             v                  v
Variable X  - - - - - - - -  Variable Y
         (asociación observada, no necesariamente causal)
```

El análisis exploratorio **genera hipótesis**; para contrastarlas se requieren diseños experimentales o métodos de inferencia causal (Pearl y Mackenzie, 2018).

## 14.1 Paradoja de Simpson

Una tendencia observada en el conjunto total puede **invertirse** al analizar subgrupos. El siguiente ejemplo utiliza **datos simulados** con fines ilustrativos: dentro de cada grupo la relación entre X e Y es negativa, pero al agrupar todas las observaciones la relación aparenta ser positiva.

In [ ]:
rng = np.random.default_rng(7)
bloques = []
for nombre, centro in zip(["Grupo A", "Grupo B", "Grupo C"], [2, 5, 8]):
    x_g = rng.normal(centro, 0.8, 60)
    y_g = centro - 0.7 * (x_g - centro) + rng.normal(0, 0.5, 60)
    bloques.append(pd.DataFrame({"x": x_g, "y": y_g, "grupo": nombre}))
simpson = pd.concat(bloques, ignore_index=True)

print(f"Correlación global: {simpson['x'].corr(simpson['y']):+.2f}")
for nombre, sub in simpson.groupby("grupo"):
    print(f"Correlación dentro de {nombre}: {sub['x'].corr(sub['y']):+.2f}")

plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=simpson, x="x", y="y", hue="grupo", alpha=0.8)
sns.regplot(data=simpson, x="x", y="y", scatter=False, color="black", ci=None, line_kws={"linestyle": "--"})
for nombre, sub in simpson.groupby("grupo"):
    sns.regplot(data=sub, x="x", y="y", scatter=False, ci=None, line_kws={"linewidth": 1.5})
plt.title("Paradoja de Simpson (datos simulados)")
plt.tight_layout()
plt.show()

### Lección metodológica

La relación global y las relaciones dentro de los grupos pueden contar historias opuestas. Por ello, cuando existe una variable de agrupación plausible (género, categoría de IMC, ocupación), es necesario verificar si la asociación se mantiene dentro de cada subgrupo antes de generalizar.

# 15. Correlación parcial: controlar el efecto de otras variables

La **correlación parcial** mide la asociación entre X e Y una vez eliminada la influencia lineal de una o más variables de control Z. El procedimiento es:

1. Ajustar una regresión lineal de X sobre Z y conservar los residuos de X.
2. Ajustar una regresión lineal de Y sobre Z y conservar los residuos de Y.
3. Calcular la correlación de Pearson entre ambos conjuntos de residuos.

Los residuos representan la parte de cada variable que **no es explicada** por las variables de control.

**Interpretación comparativa:**

- Si la correlación parcial es similar a la simple, las variables de control no explican la asociación.
- Si la correlación parcial es mucho menor, parte de la asociación observada se debe a la influencia de Z.
- Si cambia de signo, la tercera variable modifica sustancialmente la lectura del fenómeno.

In [ ]:
def correlacion_parcial(datos, x, y, controles):
    """Correlación entre x e y tras remover linealmente el efecto de 'controles'."""
    sub = datos[[x, y] + controles].dropna()
    Z = np.column_stack([np.ones(len(sub))] + [sub[c].to_numpy(float) for c in controles])

    def residuos(variable):
        valores = sub[variable].to_numpy(float)
        coeficientes, *_ = np.linalg.lstsq(Z, valores, rcond=None)
        return valores - Z @ coeficientes

    return np.corrcoef(residuos(x), residuos(y))[0, 1]


casos = [
    ("Physical Activity Level", "Quality of Sleep", ["Stress Level"]),
    ("Sleep Duration", "Quality of Sleep", ["Stress Level"]),
    ("Sleep Duration", "Quality of Sleep", ["Stress Level", "Physical Activity Level"]),
    ("Age", "Heart Rate", ["Physical Activity Level", "Daily Steps"]),
]

filas = []
for x_var, y_var, ctrl in casos:
    filas.append(
        {
            "X": x_var,
            "Y": y_var,
            "Controles": ", ".join(ctrl),
            "r simple": df[x_var].corr(df[y_var]),
            "r parcial": correlacion_parcial(df, x_var, y_var, ctrl),
        }
    )

parciales = pd.DataFrame(filas)
parciales["Cambio"] = parciales["r parcial"] - parciales["r simple"]
parciales.round(3)

### Interpretación

Para cada fila se debe responder: ¿la correlación parcial es mayor, menor o de signo distinto respecto a la simple? ¿Qué sugiere esa diferencia sobre el papel de las variables de control? Cabe precisar que el control estadístico es **lineal** y solo considera las variables incluidas: no elimina la posible confusión producida por variables no medidas.

# 16. Matriz de dispersión (pairplot)

El `pairplot` presenta en una sola figura las distribuciones individuales (diagonal) y las relaciones entre todos los pares (fuera de la diagonal). Es una herramienta de **exploración inicial** que permite detectar formas de relación, agrupamientos y atípicos. Al incorporar una variable categórica como color, se añade una tercera dimensión de lectura.

In [ ]:
variables_pairplot = [
    "Age",
    "Sleep Duration",
    "Quality of Sleep",
    "Stress Level",
    "Physical Activity Level",
]

sns.pairplot(
    df[variables_pairplot + ["Sleep Disorder"]],
    hue="Sleep Disorder",
    diag_kind="hist",
    corner=True,
    plot_kws={"alpha": 0.6, "s": 25},
    height=2.0,
)
plt.show()

### Actividad

Seleccionar **dos relaciones** del gráfico y, para cada una:

1. Identificar las variables involucradas.
2. Indicar si la asociación parece positiva, negativa o débil.
3. Evaluar si es aproximadamente lineal o presenta otra forma.
4. Señalar si los grupos de `Sleep Disorder` se separan o se mezclan.
5. Indicar qué análisis adicional se realizaría.

No se debe describir todos los gráficos: corresponde seleccionar los resultados relevantes y justificar la selección.

# 17. Variable numérica según una variable categórica

Cuando una de las variables es categórica, la pregunta pasa de "¿varían juntas?" a "¿el nivel de la variable numérica **difiere entre grupos**?". El análisis se basa en estadísticos por grupo y en gráficos que muestran la distribución completa.

**Recomendación:** reportar siempre el tamaño de cada grupo (`n`). Una diferencia de medias entre grupos muy desiguales puede deberse a la inestabilidad de los grupos pequeños.

In [ ]:
orden_imc = [c for c in ["Normal", "Overweight", "Obese"] if c in df["BMI Category"].unique()]
orden_trastorno = [c for c in ["None", "Insomnia", "Sleep Apnea"] if c in df["Sleep Disorder"].unique()]

resumen_grupos = (
    df.groupby("Sleep Disorder")[["Sleep Duration", "Quality of Sleep", "Stress Level"]]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
resumen_grupos

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.5))

for eje, variable in zip(ejes, ["Sleep Duration", "Quality of Sleep", "Stress Level"]):
    sns.boxplot(data=df, x="Sleep Disorder", y=variable, order=orden_trastorno, ax=eje, color="#9ECAE9")
    sns.stripplot(data=df, x="Sleep Disorder", y=variable, order=orden_trastorno, ax=eje,
                  color="black", alpha=0.25, size=2.5, jitter=0.25)
    eje.set_title(f"{variable} según trastorno del sueño")
    eje.set_xlabel("Trastorno del sueño")

plt.tight_layout()
plt.show()

### Lectura del diagrama de cajas

- La **línea central** es la mediana; los **bordes de la caja** son el primer y tercer cuartil (el 50 % central de los datos).
- Los **bigotes** se extienden hasta 1.5 veces el rango intercuartílico; los puntos fuera se consideran atípicos según el criterio de Tukey.
- Los puntos superpuestos permiten visualizar el tamaño y la densidad de cada grupo, información que la caja por sí sola oculta.

### Correlación dentro de subgrupos

Para verificar si una asociación se mantiene en todos los grupos (idea de la paradoja de Simpson), se calcula la correlación por subgrupo.

In [ ]:
def correlacion_por_grupo(datos, variable_grupo, x, y, n_minimo=10):
    filas = [{variable_grupo: "Todos", "n": len(datos), "r": datos[x].corr(datos[y])}]
    for nombre, sub in datos.groupby(variable_grupo):
        if len(sub) >= n_minimo:
            filas.append({variable_grupo: nombre, "n": len(sub), "r": sub[x].corr(sub[y])})
    return pd.DataFrame(filas)


for grupo in ["Gender", "BMI Category"]:
    print(f"Correlación entre Sleep Duration y Quality of Sleep según {grupo}")
    display(correlacion_por_grupo(df, grupo, "Sleep Duration", "Quality of Sleep").round(3))

# 18. Asociación entre dos variables categóricas

La correlación de Pearson no es aplicable a variables nominales. El análisis se basa en la **tabla de contingencia**, que cruza las frecuencias de las categorías.

### Elementos del análisis

1. **Frecuencias observadas:** conteos reales en cada combinación.
2. **Proporciones por fila:** distribución de una variable dentro de cada categoría de la otra (permite comparar grupos de distinto tamaño).
3. **Frecuencias esperadas:** conteos que se observarían si las variables fueran independientes, calculados como (total de fila × total de columna) / total general.
4. **Prueba chi-cuadrado de independencia:** compara observadas y esperadas.
5. **V de Cramér:** medida de la intensidad de la asociación, independiente del tamaño de muestra:

$$ V = \sqrt{\frac{\chi^2}{n \cdot \min(k-1,\, c-1)}} $$

donde $k$ y $c$ son el número de filas y columnas. Varía entre 0 (independencia) y 1 (asociación perfecta).

In [ ]:
tabla = pd.crosstab(df["BMI Category"], df["Sleep Disorder"])
proporciones = pd.crosstab(df["BMI Category"], df["Sleep Disorder"], normalize="index")

print("Frecuencias observadas")
display(tabla)
print("Proporciones por fila (cada fila suma 1)")
display(proporciones.round(3))

In [ ]:
chi2, p_valor, grados_libertad, esperadas = stats.chi2_contingency(tabla)
n_total = tabla.to_numpy().sum()
v_cramer = np.sqrt(chi2 / (n_total * (min(tabla.shape) - 1)))
proporcion_esperadas_bajas = (esperadas < 5).mean()

print(f"Chi-cuadrado: {chi2:.2f}  |  grados de libertad: {grados_libertad}  |  valor p: {p_valor:.3e}")
print(f"V de Cramér: {v_cramer:.3f}")
print(f"Proporción de celdas con frecuencia esperada menor que 5: {proporcion_esperadas_bajas:.0%}")

plt.figure(figsize=(7, 4.5))
sns.heatmap(proporciones, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            cbar_kws={"label": "Proporción dentro de la categoría de IMC"})
plt.title("Trastorno del sueño según categoría de IMC")
plt.tight_layout()
plt.show()

### Interpretación y supuestos

- El valor p evalúa si existe evidencia de asociación; la **V de Cramér** informa cuán fuerte es. Ambos deben reportarse.
- La prueba chi-cuadrado requiere que las frecuencias esperadas sean suficientemente grandes (criterio habitual: al menos 5 en la mayoría de las celdas). Si se incumple, se deben agrupar categorías o recurrir a pruebas exactas.
- Una asociación entre categorías no indica causalidad ni la dirección de la relación.

# 19. Incorporación de una tercera variable: canales visuales y facetas

Una visualización puede codificar variables adicionales mediante distintos **canales visuales**: color, tamaño, forma del marcador y división en paneles (facetas).

| Canal | Adecuado para | Observación |
|---|---|---|
| Posición (ejes) | Variables numéricas principales | Es el canal más preciso para la percepción humana |
| Color | Categorías o una variable ordinal | Usar paletas secuenciales para ordinales |
| Tamaño | Variable numérica secundaria | Percepción menos precisa que la posición |
| Forma | Categorías pocas (2 o 3) | Más de 4 formas dificulta la lectura |
| Facetas | Comparar subgrupos | Evita la superposición y facilita comparar patrones |

**Principio de parsimonia visual:** cada canal adicional incrementa la carga cognitiva. Se recomienda no superar tres o cuatro codificaciones simultáneas.

Pregunta:

> **¿La relación entre actividad física y calidad del sueño es igual en todos los niveles de estrés?**

In [ ]:
rng = np.random.default_rng(11)
jitter = rng.uniform(-0.15, 0.15, size=len(df))

plt.figure(figsize=(10, 6.5))
sns.scatterplot(
    x=df["Physical Activity Level"],
    y=df["Quality of Sleep"] + jitter,
    hue=df["Stress Level"],
    size=df["Heart Rate"],
    style=df["Gender"],
    palette="viridis",
    sizes=(25, 140),
    alpha=0.75,
)
plt.title("Cinco variables en una sola vista: actividad, calidad, estrés, frecuencia cardíaca y género")
plt.xlabel("Actividad física (minutos por día)")
plt.ylabel("Calidad del sueño (con dispersión visual)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

El gráfico anterior codifica cinco variables y ilustra el riesgo de sobrecarga visual: resulta difícil extraer conclusiones precisas. Una alternativa más legible es dividir los datos en **facetas** según niveles de estrés y ajustar una recta de tendencia en cada panel.

In [ ]:
df["Nivel de estrés"] = pd.cut(
    df["Stress Level"],
    bins=[0, 3, 6, 10],
    labels=["Bajo (hasta 3)", "Medio (4 a 6)", "Alto (7 o más)"],
)
niveles_presentes = [n for n in df["Nivel de estrés"].cat.categories if (df["Nivel de estrés"] == n).any()]

g = sns.lmplot(
    data=df,
    x="Physical Activity Level",
    y="Quality of Sleep",
    col="Nivel de estrés",
    col_order=niveles_presentes,
    height=4,
    scatter_kws={"alpha": 0.4, "s": 25},
    line_kws={"color": "#E45756"},
)
g.set_axis_labels("Actividad física (min/día)", "Calidad del sueño")
g.figure.suptitle("Actividad física y calidad del sueño según nivel de estrés", y=1.03)
plt.show()

print(correlacion_por_grupo(df.assign(grupo=df["Nivel de estrés"].astype(str)), "grupo",
                            "Physical Activity Level", "Quality of Sleep").round(3))

### Interpretación

Se compara la relación **global** entre actividad física y calidad del sueño con las relaciones **dentro de cada nivel de estrés**:

- ¿La pendiente y el coeficiente se mantienen, se debilitan o cambian de signo en los subgrupos?
- ¿Algún panel tiene pocas observaciones que hagan inestable la estimación?
- ¿Existe evidencia para afirmar causalidad? ¿Qué diseño de estudio la proporcionaría?

# 20. Extensión A: valores atípicos multivariantes

Un valor puede no ser extremo en ninguna variable individual y, sin embargo, resultar **atípico en la combinación**. Por ejemplo, una persona con 8 horas de sueño y otra con calidad de sueño muy baja son perfiles posibles por separado, pero la combinación de ambos rasgos puede ser inusual respecto al patrón conjunto.

La **distancia de Mahalanobis** mide qué tan lejos está una observación del centro multivariante, considerando la escala y la correlación entre variables:

$$ D^2_i = (\mathbf{x}_i - \boldsymbol{\mu})^{\top} \, \mathbf{S}^{-1} \, (\mathbf{x}_i - \boldsymbol{\mu}) $$

donde $\boldsymbol{\mu}$ es el vector de medias y $\mathbf{S}$ la matriz de covarianzas. Bajo normalidad multivariante, $D^2$ sigue aproximadamente una distribución chi-cuadrado con $p$ grados de libertad (p = número de variables), lo que permite establecer un umbral.

**Advertencia:** los datos con escalas discretas o asimétricas incumplen la normalidad; el umbral es una **referencia exploratoria**, y las observaciones señaladas deben revisarse, no eliminarse de forma automática.

In [ ]:
X = df[variables_numericas].to_numpy(dtype=float)
centro = X.mean(axis=0)
S = np.cov(X, rowvar=False)
S_inversa = np.linalg.pinv(S)  # pseudoinversa: robusta ante casi-singularidad

desviaciones = X - centro
d2 = np.einsum("ij,jk,ik->i", desviaciones, S_inversa, desviaciones)

umbral = stats.chi2.ppf(0.975, df=len(variables_numericas))
df["D2 Mahalanobis"] = d2
atipicos = df.loc[df["D2 Mahalanobis"] > umbral]

print(f"Umbral (chi-cuadrado, p = 0.975, gl = {len(variables_numericas)}): {umbral:.2f}")
print(f"Observaciones por encima del umbral: {len(atipicos)} de {len(df)} ({len(atipicos) / len(df):.1%})")

plt.figure(figsize=(8, 4.5))
plt.hist(d2, bins=30, color="#9ECAE9", edgecolor="white")
plt.axvline(umbral, color="#E45756", linestyle="--", label=f"Umbral = {umbral:.1f}")
plt.title("Distribución de la distancia de Mahalanobis al cuadrado")
plt.xlabel("D2")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()

atipicos[variables_numericas + ["D2 Mahalanobis"]].sort_values("D2 Mahalanobis", ascending=False).head(10).round(2)

### Criterios para tratar atípicos

1. **Verificar** si se trata de un error de captura o transcripción.
2. **Evaluar** si es un caso legítimo pero poco frecuente (que podría ser el más informativo).
3. **Medir la sensibilidad:** comparar los resultados con y sin esas observaciones.
4. **Documentar** la decisión. Eliminar datos sin justificación compromete la integridad del análisis.

# 21. Extensión B: multicolinealidad

Existe **multicolinealidad** cuando varias variables explicativas están fuertemente correlacionadas entre sí. En un análisis exploratorio señala redundancia; en modelos posteriores (regresión) genera coeficientes inestables y difíciles de interpretar.

El **factor de inflación de la varianza (VIF)** cuantifica la redundancia de cada variable respecto a las demás:

$$ \text{VIF}_j = \frac{1}{1 - R_j^2} $$

donde $R_j^2$ es el coeficiente de determinación al predecir la variable $j$ con todas las restantes.

| VIF | Lectura orientativa |
|---:|---|
| 1 | Sin redundancia |
| Entre 1 y 5 | Redundancia moderada, generalmente aceptable |
| Entre 5 y 10 | Redundancia importante, requiere atención |
| Mayor que 10 | Redundancia severa |

Los umbrales son convencionales y dependen del contexto.

In [ ]:
def calcular_vif(datos, columnas):
    matriz = datos[columnas].to_numpy(dtype=float)
    filas = []
    for j, columna in enumerate(columnas):
        objetivo = matriz[:, j]
        predictores = np.column_stack([np.ones(len(matriz)), np.delete(matriz, j, axis=1)])
        coeficientes, *_ = np.linalg.lstsq(predictores, objetivo, rcond=None)
        residuos = objetivo - predictores @ coeficientes
        r2 = 1 - residuos.var() / objetivo.var()
        filas.append({"Variable": columna, "R2 frente a las demás": r2, "VIF": 1 / (1 - r2)})
    return pd.DataFrame(filas).sort_values("VIF", ascending=False)


calcular_vif(df, variables_numericas).round(3).reset_index(drop=True)

### Interpretación

Las variables con VIF alto aportan información que ya está contenida en otras. Ante esta situación existen tres alternativas: conservar una sola de las variables redundantes, combinarlas en un índice o resumirlas mediante componentes principales (sección siguiente).

# 22. Extensión C: reducción de dimensionalidad con PCA

El **análisis de componentes principales (PCA)** transforma un conjunto de variables correlacionadas en nuevas variables no correlacionadas, llamadas **componentes principales**, ordenadas por la cantidad de variabilidad que capturan.

### Conceptos clave

- **Estandarización previa:** las variables tienen unidades distintas (años, pasos, latidos). Sin estandarizar, las de mayor escala dominarían los componentes. Por ello se trabaja con la matriz de correlación (variables con media 0 y desviación 1).
- **Autovalor:** varianza capturada por cada componente. Con variables estandarizadas, la suma de autovalores es igual al número de variables.
- **Varianza explicada:** proporción de la variabilidad total que recoge cada componente.
- **Cargas (loadings):** peso de cada variable original en cada componente; permiten interpretar su significado.
- **Criterios para decidir cuántos componentes retener (orientativos):** autovalor mayor que 1 (criterio de Kaiser), varianza acumulada de 70 % a 80 %, y el punto de inflexión del gráfico de sedimentación (scree plot).

### Limitaciones

PCA es lineal, sensible a atípicos y a la escala, y sus componentes son combinaciones matemáticas que no siempre tienen interpretación sustantiva clara.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Z = StandardScaler().fit_transform(df[variables_numericas])
pca = PCA().fit(Z)

nombres_pc = [f"PC{i + 1}" for i in range(len(variables_numericas))]
resumen_pca = pd.DataFrame(
    {
        "Componente": nombres_pc,
        "Autovalor": pca.explained_variance_,
        "Varianza explicada": pca.explained_variance_ratio_,
        "Varianza acumulada": np.cumsum(pca.explained_variance_ratio_),
    }
)
display(resumen_pca.round(3))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))

ejes[0].bar(nombres_pc, pca.explained_variance_ratio_, color="#9ECAE9", label="Individual")
ejes[0].plot(nombres_pc, np.cumsum(pca.explained_variance_ratio_), marker="o", color="#E45756", label="Acumulada")
ejes[0].set_title("Varianza explicada por componente")
ejes[0].set_ylabel("Proporción de varianza")
ejes[0].legend()

ejes[1].plot(nombres_pc, pca.explained_variance_, marker="o", color="#4C78A8")
ejes[1].axhline(1, color="gray", linestyle="--", label="Criterio de Kaiser (autovalor = 1)")
ejes[1].set_title("Gráfico de sedimentación")
ejes[1].set_ylabel("Autovalor")
ejes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
cargas = pd.DataFrame(pca.components_.T, index=variables_numericas, columns=nombres_pc)
print("Cargas de las variables en los tres primeros componentes")
display(cargas[["PC1", "PC2", "PC3"]].round(2))

proyeccion = pd.DataFrame(pca.transform(Z)[:, :2], columns=["PC1", "PC2"])
proyeccion["Sleep Disorder"] = df["Sleep Disorder"].to_numpy()

plt.figure(figsize=(8, 6))
sns.scatterplot(data=proyeccion, x="PC1", y="PC2", hue="Sleep Disorder",
                hue_order=orden_trastorno, alpha=0.7)
plt.axhline(0, color="gray", linewidth=0.6)
plt.axvline(0, color="gray", linewidth=0.6)
plt.title("Observaciones proyectadas en los dos primeros componentes")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} de la varianza)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} de la varianza)")
plt.tight_layout()
plt.show()

### Interpretación

1. **Cuántos componentes retener:** aplicar los criterios de Kaiser, varianza acumulada y gráfico de sedimentación, y justificar la decisión.
2. **Significado de cada componente:** identificar las variables con cargas absolutas altas (por ejemplo, mayores que 0.4) y proponer una interpretación conceptual (una dimensión de "carga de estrés y sueño", una de "actividad física", etc.).
3. **Separación de grupos:** evaluar si los grupos de `Sleep Disorder` se distinguen en el plano PC1 y PC2. Una separación visible sugiere que las variables numéricas contienen información relacionada con esa categoría; una mezcla sugiere lo contrario.
4. **Precaución:** el PCA no utiliza `Sleep Disorder` para construir los componentes. Es un método **no supervisado**; el color solo se usa después para interpretar.

# 23. Práctica autónoma: construcción de un análisis multivariante

Se solicita formular una pregunta que requiera **al menos tres variables** y desarrollar un análisis completo.

### Líneas temáticas sugeridas

- Sueño y bienestar (duración, calidad, estrés).
- Actividad física y sueño (actividad, pasos, calidad).
- Estrés y características fisiológicas (estrés, frecuencia cardíaca, IMC).
- Edad y hábitos.
- Diferencias entre grupos (género, ocupación, trastorno del sueño).

### Contenido obligatorio del análisis

| N.º | Componente | Descripción |
|---:|---|---|
| 1 | Pregunta exploratoria | Formulada con claridad antes de elegir el gráfico |
| 2 | Variables seleccionadas | Con su tipo y escala de medición |
| 3 | Justificación | Por qué esas variables responden la pregunta |
| 4 | Técnica | Coherente con el tipo de variables (sección 3) |
| 5 | Evidencia numérica | Coeficiente, tabla o estadístico, con medida de incertidumbre cuando corresponda |
| 6 | Visualización | Al menos una, con título, ejes y unidades |
| 7 | Interpretación | Redactada con la plantilla de la sección 24 |
| 8 | Limitaciones | Al menos dos (por ejemplo, origen de los datos, causalidad, escalas ordinales) |
| 9 | Nueva pregunta | Surgida del propio análisis |

> **Principio:** no se debe iniciar por el gráfico, sino por la pregunta: ¿qué se desea descubrir y qué evidencia permitiría sostenerlo?

### Registro de la pregunta y del plan analítico

Completar en esta celda:

- **Pregunta exploratoria:** 
- **Variables y escalas:** 
- **Técnica elegida y justificación:** 
- **Resultado esperado (hipótesis exploratoria):** 

In [ ]:
# Selección y preparación de las variables del análisis propio.

In [ ]:
# Cálculo de la evidencia numérica (correlaciones, tablas, estadísticos).

In [ ]:
# Construcción de la visualización.

### Registro de hallazgos

Completar en esta celda: hallazgo principal, evidencia que lo respalda, interpretación, limitaciones y nueva pregunta.

# 24. Comunicación de hallazgos y criterios de evaluación

## 24.1 Plantilla para comunicar un hallazgo

Separar de forma explícita **resultado**, **interpretación** y **afirmación causal** evita errores de comunicación.

| Componente | Estructura sugerida |
|---|---|
| Evidencia | "La correlación entre X e Y es r = ... (IC 95 %: ... a ...; n = ...)." |
| Dirección e intensidad | "La asociación es positiva (negativa) y, según la escala orientativa utilizada, de intensidad ..." |
| Magnitud práctica | "X comparte aproximadamente el ... % de su variabilidad con Y (r al cuadrado)." |
| Interpretación contextual | "En estos registros, cuando X aumenta, Y tiende a ..." |
| Contraste gráfico | "El diagrama de dispersión confirma (o matiza) este resultado porque ..." |
| Precaución | "Este resultado no permite afirmar que X cause Y porque ..." |
| Siguiente pregunta | "Sería pertinente investigar ..." |

## 24.2 Rúbrica de evaluación de la práctica

| Criterio | Descripción del desempeño esperado | Ponderación |
|---|---|---:|
| Formulación de la pregunta | Pregunta clara, acotada y que justifica el uso de varias variables | 15 % |
| Selección de variables y técnica | Coherencia entre tipo de variable y técnica elegida | 15 % |
| Rigor numérico | Cálculos correctos, con medida de incertidumbre o tamaño de efecto | 20 % |
| Visualización | Gráfico apropiado, legible, con título, ejes y unidades | 15 % |
| Interpretación | Distingue dirección, intensidad y contexto sin sobreinterpretar | 20 % |
| Limitaciones y pensamiento crítico | Identifica al menos dos limitaciones pertinentes y una nueva pregunta | 15 % |
| **Total** | | **100 %** |

# 25. Errores frecuentes de interpretación

| Formulación incorrecta | Problema | Formulación adecuada |
|---|---|---|
| "r = -0.75, por lo tanto la relación es mala." | Confunde el signo con la calidad de la relación | "Existe una asociación lineal negativa fuerte." |
| "r = 0.10, por lo tanto no existe ninguna relación." | Excesivamente categórica; ignora relaciones no lineales | "La asociación lineal observada es muy débil; se debe revisar el gráfico." |
| "r = 0.85, por lo tanto X causa Y." | Confunde correlación con causalidad | "X e Y presentan una asociación lineal fuerte en estos datos." |
| "El valor p es 0.001, por lo tanto la relación es muy importante." | Confunde significancia estadística con relevancia práctica | "La asociación es estadísticamente significativa; su magnitud es r = ..." |
| "Se analizaron muchas variables, por eso el análisis es multivariante." | La cantidad no sustituye al propósito analítico | "Se seleccionaron variables que juntas responden una pregunta específica." |
| "La correlación global es positiva, por lo tanto lo es en todos los grupos." | Ignora la paradoja de Simpson | "Se verificó la asociación dentro de cada subgrupo." |
| "Se eliminaron los atípicos para mejorar el resultado." | Manipulación sin justificación | "Se evaluó la sensibilidad del resultado con y sin las observaciones señaladas." |
| "Pearson es siempre la medida correcta." | Ignora escalas ordinales y relaciones no lineales | "Se compararon Pearson y Spearman y se inspeccionó el gráfico." |

# 26. Síntesis y glosario

```text
                         PREGUNTA
                            |
                  Selección de variables
                            |
                 Auditoría de calidad de datos
                            |
          +-----------------+-----------------+
          |                 |                 |
     Univariante        Bivariante       Multivariante
          |                 |                 |
      Describe          Relaciona        Contextualiza
          +-----------------+-----------------+
                            |
              Visualización y evidencia numérica
                            |
                     Interpretación
                            |
                      Limitaciones
                            |
                        Hallazgo
                            |
                     Nueva pregunta
```

| Término | Definición |
|---|---|
| Análisis multivariante | Estudio simultáneo de tres o más variables |
| Covarianza | Medida de variación conjunta, dependiente de las unidades |
| Correlación de Pearson | Asociación lineal estandarizada, entre -1 y +1 |
| Correlación de Spearman | Asociación monótona basada en rangos |
| Coeficiente de determinación ($r^2$) | Proporción de variabilidad compartida linealmente |
| Valor p | Probabilidad de obtener un resultado igual o más extremo si $H_0$ fuera cierta |
| Intervalo de confianza | Rango de valores plausibles para el parámetro poblacional |
| Variable de confusión | Variable que influye en X e Y y genera o distorsiona su asociación |
| Paradoja de Simpson | Inversión de una tendencia al analizar subgrupos |
| Correlación parcial | Correlación entre X e Y tras controlar el efecto de otras variables |
| V de Cramér | Intensidad de asociación entre variables categóricas |
| Distancia de Mahalanobis | Distancia al centro multivariante que considera la covarianza |
| Multicolinealidad | Redundancia lineal entre variables explicativas |
| VIF | Factor de inflación de la varianza, indicador de multicolinealidad |
| PCA | Técnica para resumir variables correlacionadas en pocos componentes |

> **El objetivo del análisis exploratorio no es encontrar una respuesta definitiva, sino comprender los datos lo suficiente para formular mejores preguntas y tomar decisiones analíticas fundamentadas.**

# 27. Lista de verificación de aprendizaje

- [ ] Diferencio los análisis univariante, bivariante y multivariante, y sé cuándo corresponde cada uno.
- [ ] Selecciono la técnica según la escala de medición de las variables.
- [ ] Calculo e interpreto covarianza, Pearson y Spearman.
- [ ] Interpreto dirección mediante el signo e intensidad mediante $|r|$, reconociendo que los umbrales son orientativos.
- [ ] Interpreto $r^2$ como variabilidad compartida.
- [ ] Leo una matriz de correlación y selecciono los pares relevantes.
- [ ] Explico por qué un coeficiente debe acompañarse de un gráfico (cuarteto de Anscombe).
- [ ] Distingo significancia estadística de relevancia práctica.
- [ ] Interpreto un intervalo de confianza para una correlación.
- [ ] Reconozco variables de confusión y la paradoja de Simpson.
- [ ] Aplico correlación parcial y análisis por subgrupos.
- [ ] Analizo variables categóricas con tablas de contingencia, chi-cuadrado y V de Cramér.
- [ ] Comprendo el propósito de la distancia de Mahalanobis, del VIF y del PCA.
- [ ] No confundo correlación con causalidad.
- [ ] Comunico un hallazgo con evidencia, interpretación y limitaciones.

# 28. Referencias bibliográficas

- Anscombe, F. J. (1973). Graphs in statistical analysis. *The American Statistician, 27*(1), 17-21.
- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2.ª ed.). Lawrence Erlbaum Associates.
- Hair, J. F., Black, W. C., Babin, B. J. y Anderson, R. E. (2019). *Multivariate Data Analysis* (8.ª ed.). Cengage.
- James, G., Witten, D., Hastie, T. y Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2.ª ed.). Springer.
- Johnson, R. A. y Wichern, D. W. (2007). *Applied Multivariate Statistical Analysis* (6.ª ed.). Pearson.
- McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly Media.
- Pearl, J. y Mackenzie, D. (2018). *The Book of Why: The New Science of Cause and Effect*. Basic Books.
- Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley.
- VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly Media.
- Wickham, H., Çetinkaya-Rundel, M. y Grolemund, G. (2023). *R for Data Science* (2.ª ed.). O'Reilly Media.

# Anexo para el docente: solucionario y orientaciones

## Ejercicio de la sección 9 (clasificación de coeficientes)

| r | Dirección | Intensidad orientativa |
|---:|---|---|
| 0.78 | Positiva | Fuerte |
| -0.83 | Negativa | Muy fuerte |
| 0.31 | Positiva | Débil |
| -0.08 | Negativa | Muy débil |
| 0.52 | Positiva | Moderada |

**Sobre $+0.78$ y $-0.78$:** la intensidad depende de $|r|$, que es igual en ambos casos (0.78); el signo solo indica el sentido de la variación conjunta. Un coeficiente negativo no implica una relación "mala": indica que, cuando una variable aumenta, la otra tiende a disminuir, lo cual puede ser perfectamente esperable (por ejemplo, estrés y calidad del sueño).

## Orientaciones por sección

| Sección | Punto de discusión sugerido |
|---|---|
| 5 | La naturaleza posiblemente sintética de los datos limita la generalización; debe constar en las limitaciones |
| 6 | Justificar la decisión sobre duplicados; la categoría `None` no es un valor faltante |
| 7 | Relacionar la superposición de puntos con la escala discreta de las variables |
| 9 | Insistir en que $r^2$ es menor que $r$ en valor absoluto: una correlación "moderada" comparte poco de la variabilidad |
| 11 | Si Pearson y Spearman difieren, la causa debe verificarse gráficamente |
| 12 | Los cuatro conjuntos de Anscombe comparten estadísticos; solo el gráfico los diferencia |
| 13 | La simulación muestra que el valor p depende del tamaño de muestra, no de la importancia del efecto |
| 14 | En la paradoja de Simpson, la correlación global es positiva y las de los grupos son negativas |
| 15 | Comparar r simple y r parcial; discutir qué variable de confusión podría explicar la diferencia |
| 18 | Verificar el supuesto de frecuencias esperadas antes de interpretar chi-cuadrado |
| 20 a 22 | Bloque de extensión: asignar como trabajo autónomo si el tiempo de clase es insuficiente |

## Sugerencias de evaluación complementaria

1. Solicitar a los estudiantes que identifiquen **un resultado del notebook que no sustentaría una conclusión causal** y justifiquen por qué.
2. Pedir que repliquen el análisis de correlación parcial con otra variable de control y comparen el resultado.
3. Plantear una pregunta de discusión: *si la muestra fuera aleatoria y de una población real, ¿qué cambiaría en la interpretación de los valores p?*